In [ ]:
from embedder import Embedder

embed = Embedder()

In [ ]:
q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

In [ ]:
v1.dot(dv)

In [ ]:
v2.dot(dv)

In [ ]:
v3 = embed.encode("How does approximate nearest neighbor search work?")

In [ ]:
v3[0]

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [ ]:
target_user = next((item for item in documents if item["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"), None)


In [ ]:
v4 = embed.encode(target_user['content'])


In [ ]:
v3.dot(v4)

In [ ]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
print(len(documents))

In [ ]:
print(len(chunks))

In [ ]:
texts = [doc["filename"] + " " + doc["content"] for doc in chunks]

In [ ]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

In [ ]:
scores = X.dot(v3)


In [ ]:
scores.shape

In [ ]:
idx = np.argmax(scores)
idx, scores[idx]

In [ ]:
chunks[idx]

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["lessons"])
vindex.fit(X, chunks)

In [ ]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

In [ ]:
results

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [ ]:
text_results = index.search("How do I store vectors in PostgreSQL?", num_results=5)
text_results

In [ ]:
query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

In [ ]:
results

In [41]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
query = "How do I give the model access to tools?"
text_results = index.search(query, num_results=5)
